# Relax structure (MLP)

Relax atomic positions locally with ASE using **MACE-MP** today. Quantum ESPRESSO jobs are **not** used.

Authenticate once to **load** a Mat3ra material by name (or uploads + Standata) and **save** the relaxed structure back.

<h2 style="color:green">Usage</h2>

1. Set paths, material name, relax force tolerance, MACE presets, and tagging in sections **1.1–1.3**.
1. Authenticate (`OIDC_ACCESS_TOKEN`) and bind `ACCOUNT_ID`.
1. Local JSON lives under `../uploads`; platform mode searches by `MATERIAL_NAME`.
1. Click "Run" > "Run All".

## Summary

Install packages (+ PyTorch quirks in JupyterLite), pull material locally or API, visualize, optimize with ASE + MACE (+ `notebooks_utils` Plotly progress helpers), analyse optional heterostructure metrics, upload relaxed geometry.

**Switching calculators later** (different MLP wheels, constructor args): change the installer string in §1.1 and the calculator block in §4.1—the rest of the workflow stays ASE-shaped.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages
from mat3ra.notebooks_utils.primitive.environment import is_pyodide_environment

await install_packages("made|api_examples|torch|mace")

if is_pyodide_environment():
    from mat3ra.notebooks_utils.pyodide.packages.torch import apply_all_patches

    apply_all_patches()

### 1.2. Set parameters

In [ ]:
ORGANIZATION_NAME = None

FOLDER = "../uploads"
MATERIAL_NAME = "S2W(001)-Al2O3(001), Interface, Strain 0.311pct"  # used for local/Standata lookup and as platform search term


### 1.3. Relaxation and MACE options

In [ ]:
# Final maximum force on any atom (eV/Å)
FMAX = 0.05

MACE_MODEL = "large"  # choose between "small", "medium", and "large"
MACE_DISPERSION = True  # enable D3 dispersion correction for van der Waals interactions
MACE_DEFAULT_DTYPE = "float32"  # floating-point precision for model inference; "float64" is more accurate but slower
MACE_DEVICE = "cpu"  # hardware target: "cpu" or "cuda" (GPU)

## 2. Authenticate and initialize API client
### 2.1. Authenticate

Browser login stores credentials in `OIDC_ACCESS_TOKEN`.

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"\u2705 Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

## 3. Load material
### 3.1. Resolve structure (platform API or uploads + Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import ViewersEnum, visualize_materials as visualize

platform_matches = client.materials.list({"name": MATERIAL_NAME, "owner._id": ACCOUNT_ID})
material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(platform_matches[0])

visualize([{"material": material, "title": material.name}], viewer=ViewersEnum.wave)

### 3.2. Persist locally sourced structure
Runs deduplicated uploads via structural hash comparison when loading from uploads/Standata.

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

input_saved = get_or_create_material(client, material, ACCOUNT_ID)
input_saved_material = Material.create(input_saved)
print("\u2705 Canonical input material record:", input_saved_material.id)


## 4. Apply relaxation
### 4.1. Create ASE calculator with MACE-MP foundation model

In [ ]:

from mat3ra.notebooks_utils.primitive.environment import is_pyodide_environment
from mat3ra.notebooks_utils.pyodide.packages.mace import get_mace_model_pyodide
from mat3ra.made.tools.convert import to_ase
from mace.calculators import mace_mp

mace_mp = mace_mp if not is_pyodide_environment() else get_mace_model_pyodide
calculator = mace_mp(
    model=MACE_MODEL,
    dispersion=MACE_DISPERSION,
    default_dtype=MACE_DEFAULT_DTYPE,
    device=MACE_DEVICE)

### 4.2. Relax structure with ASE and visualize convergence

In [ ]:
from ase.optimize import BFGS

ase_atoms_relaxed_in_place = to_ase(material)
ase_atoms_relaxed_in_place.set_calculator(calculator)
from mat3ra.notebooks_utils.ipython.plot._plotly import progress_callback

dyn = BFGS(ase_atoms_relaxed_in_place)

progress_callback = progress_callback(dynamic_object=dyn, value_getter=ase_atoms_relaxed_in_place.get_total_energy,
                                      value_label="Energy (eV)")

dyn.attach(progress_callback, interval=1)
dyn.run(fmax=FMAX)

### 4.3. Extract energies before and after relaxation

In [ ]:
ase_original_structure = to_ase(material)
ase_original_structure.set_calculator(calculator)
ase_relaxed_structure = ase_atoms_relaxed_in_place

original_energy = ase_original_structure.get_total_energy()
relaxed_energy = ase_atoms_relaxed_in_place.get_total_energy()

print(f"The final energy is {float(relaxed_energy):.3f} eV.")

## 5. Analyze results
### 5.1. View structure before and after relaxation

In [ ]:
from mat3ra.made.tools.convert import from_ase

material_original = Material.create(from_ase(ase_original_structure))
material_relaxed = Material.create(from_ase(ase_relaxed_structure))
material_original.name = material.name
material_relaxed.name = material.name + " pre-relaxed (MACE)"

visualize(
    [
        {"material": material_original, "title": material_original.name},
        {"material": material_relaxed, "title": material_relaxed.name},
    ],
    viewer=ViewersEnum.wave,
)

### 5.2. Add metadata

In [ ]:
# TODO: add metadata of all the settings for relax

## 6. Persist relaxed structure
### 6.1. Save relaxed geometry to Mat3ra (deduplicated)
Uses structural hashing (`get_or_create_material`) identical to QE workflow uploads.

In [ ]:
material_relaxed.basis.set_labels_from_list([])
relaxed_saved_response = get_or_create_material(client, material_relaxed, ACCOUNT_ID)

## References

[1] mat3ra-made: https://github.com/Exabyte-io/made  
[2] ASE calculators overview: https://wiki.fysik.dtu.dk/ase/ase/calculators/calculators.html  
[3] MACE-MP-0 foundation models: https://github.com/ACEsuit/mace?tab=readme-ov-file#foundation-models